# BERT and Transformers for ESG Research

A practical study notebook for applying Transformer models to environmental, social and governance (ESG) text. It moves from the domain problem and BERT's architecture to tokenization, pretrained inference, embeddings, fine-tuning, evaluation, and responsible deployment.

## Learning outcomes
- Distinguish ESG controversy detection, sentiment, pillar classification, event extraction, and materiality assessment.
- Explain what a Transformer encoder and BERT learn, and what they do not learn.
- Inspect subword tokenization, attention masks, logits, probabilities, and pooled embeddings.
- Use a finance sentiment checkpoint carefully, and design a domain-specific fine-tuning experiment.
- Evaluate models with leakage-aware splits, class-sensitive metrics, calibration, and human review.

The examples use short news headlines. A headline classifier is a screening aid, not a substitute for source verification, impact assessment, or investment judgment.

## 1. Define the ESG task before choosing a model

ESG text analytics is not one label-prediction problem. A useful annotation schema separates:

| Field | Example | What the model should answer |
|---|---|---|
| `is_event` | A regulator opens an investigation | Does the text describe a concrete event? |
| `is_controversy` | A company is fined after a toxic spill | Is there an alleged or confirmed adverse impact? |
| `pillar` | `E`, `S`, `G`, or `multiple` | Which ESG dimension(s) are implicated? |
| `event_type` | spill, labor dispute, bribery | What happened? |
| `status` | alleged, investigated, confirmed, denied | How established is the claim? |
| `sentiment` | negative, neutral, positive | What is the tone toward the target? |
| `severity` | low / medium / high | How material is the impact under a defined rubric? |

These labels are related but not interchangeable. A negative earnings story is negative in tone but is not necessarily an ESG controversy. A report that a company "denies wrongdoing" mentions wrongdoing but does not assert that wrongdoing occurred. A report about a newly adopted emissions target may be positive ESG news, not a controversy. Decide whether the target is the company, supplier, product, or an entire sector before annotation.

## 2. Transformers and BERT: a beginner's guide

### Why do we need a Transformer?

A computer cannot work directly with sentences the way a person does. It first turns text into numbers. A simple method such as counting words can notice that `spill` appears, but it does not naturally understand that **"the company caused a spill"** and **"the company denied causing a spill"** express different claims. It can also struggle when the same word has different meanings in different sentences.

A **Transformer** is a neural-network design that builds a representation of each piece of text while looking at the other pieces around it. The important idea is context: the meaning representation for one word can change depending on the words that appear with it.

### From a sentence to numbers

Before a model reads text, a **tokenizer** breaks it into tokens. Tokens are often whole words, punctuation marks, or smaller word pieces. For example, a technical or uncommon company name might be split into several pieces. The tokenizer assigns each piece an integer ID from the model's vocabulary. Those IDs are then mapped to learned numeric vectors called **embeddings**.

The model also needs some information about token order. A sentence is not just a bag of words: `workers sued company` and `company sued workers` differ. Transformer models include position information so the model can distinguish tokens that occur in different places. Special tokens such as `[CLS]` and `[SEP]` can mark the start or boundary of a sequence, depending on the model.

### Self-attention: looking at useful context

Imagine reading: **"The regulator fined Acme after its plant released toxic waste into a river."** To understand what was fined, what was released, and where it went, a reader connects words that may be several positions apart. A Transformer does a learned version of this using **self-attention**: while building a representation for each token, it can assign different amounts of attention to other tokens in the same input.

A useful analogy is a group discussion. Each token asks, "Which other tokens help clarify me?" It compares its question with the information offered by the other tokens, then combines the most relevant information. In the technical description, these are called **queries**, **keys**, and **values**:

- A **query** represents what a token is looking for.
- A **key** represents what another token can be matched on.
- A **value** represents information that can be passed along if the match is useful.

The model computes match scores, turns them into weights, and uses those weights to combine information. The precise operation is often written as $\text{Attention}(Q,K,V)=\text{softmax}(QK^T/\sqrt{d_k})V$. You do not need to memorize the equation to understand the idea: compare, weight, and combine context. Different **attention heads** can learn to focus on different kinds of relationships. The model repeats attention and small neural-network transformations through multiple layers, gradually creating richer contextual representations.

Unlike older recurrent approaches that usually process tokens one at a time, a Transformer can process the tokens in a sequence together during training. This parallelism is one reason Transformers train effectively on large text collections. Attention can still be computationally expensive for very long documents, so BERT-style models have a maximum input length.

### Why context matters in ESG text

Compare these headlines:

- "Acme denies that its plant released toxic waste."
- "Regulators confirmed that Acme's plant released toxic waste."
- "The regulator found no evidence that Acme released toxic waste."

They share many words, but **denial**, **confirmation**, and **no evidence** change the claim. Contextual representations give a model a way to use these surrounding words rather than treating each word as an isolated keyword. This does **not** guarantee that the model gets the meaning right; it must still be tested on carefully labelled ESG examples, especially for negation, quotations, allegations, and legal status.

### BERT is one kind of Transformer

The word **Transformer** names a family of model architectures, not one particular model. **BERT** is an *encoder* Transformer. During pretraining, BERT learns from large text collections by hiding some input tokens and predicting the missing tokens using context on both the left and right. This is called **masked language modeling**. The original BERT also used a next-sentence objective; later BERT-family models do not all use exactly the same pretraining recipe.

Because BERT can use both earlier and later words when representing a token, it is called **bidirectional**. In contrast, GPT-style models are generally *decoder* Transformers trained to predict the next token from earlier tokens. Encoder-decoder models such as T5 use one part to read input and another to generate output. These variants share Transformer ideas but are designed for different tasks.

### How BERT becomes an ESG classifier

Pretraining teaches a model patterns in text; it does **not** teach the model our exact definition of an ESG controversy. To classify a headline, a common BERT setup adds a small **classification head** on top of the encoder. The text is tokenized, often wrapped in `[CLS]` and `[SEP]`, and passed through the model. The contextual representation for `[CLS]` is used as a compact summary of the input. The classification head turns that summary into **logits** (unscaled scores, one per class). Softmax can turn logits into scores that sum to one, but those scores are not automatically calibrated probabilities that an event really occurred.

During **fine-tuning**, we show the model examples with human-provided labels, such as `controversy` or `not controversy`. The model adjusts its weights to reduce its errors on the training examples. Validation data helps select settings and thresholds; a separate test set estimates performance on unseen data. The quality and consistency of the labels matter as much as the model. The notebook's fine-tuning cells are optional because meaningful training needs a reviewed, representative ESG dataset.

A different option is to keep BERT **frozen**, extract its vectors, and train a simpler model such as logistic regression on those vectors. This is called using frozen embeddings; it is cheaper to train but does not adapt BERT itself to the ESG labels.

### Important limits

A Transformer finds patterns; it is not a fact-checker, a legal expert, or a causal reasoning system. It can confuse a supplier with an issuer, repeat bias in its training data, miss a new kind of event, or be overconfident about a false claim. A headline also omits context that may be present in the article or primary source. For high-impact ESG use, preserve the evidence and source, evaluate errors carefully, monitor model behavior, and keep a trained analyst involved.

## 3. Choosing a model and a task head

- **Base encoder** (`bert-base-uncased`, DistilBERT): useful general-purpose starting points. DistilBERT is smaller and often cheaper, but it is not automatically the best model for every domain.
- **Financial-language model** (for example, FinBERT): pretrained or fine-tuned on financial text. This can help with financial sentiment vocabulary, but a sentiment checkpoint is not automatically an ESG controversy classifier.
- **ESG-adapted model**: continued pretraining on relevant, legally usable ESG documents can adapt vocabulary and style; supervised fine-tuning on carefully labelled examples is still needed for a defined classification target.
- **Task head**: sequence classification for one label per document; token classification for spans; multi-label classification when a text can have multiple pillars or events; question-answering or generative models for structured extraction, with additional validation.

Model choice is an empirical decision. Compare a simple TF-IDF baseline, a general encoder, and a finance/domain checkpoint on the same held-out data and the same label definition. Include inference cost, latency, licensing, and operational constraints.

In [1]:
import random
import numpy as np
import pandas as pd
import torch
import transformers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("device:", DEVICE)

transformers: 5.20.0
torch: 2.14.1
device: mps


In [2]:
examples = pd.DataFrame([
    ("Acme Energy was fined after a wastewater leak contaminated a local river.", 1, "E", "confirmed"),
    ("Workers at the supplier allege unsafe conditions and unpaid overtime.", 1, "S", "alleged"),
    ("The regulator opened an inquiry into alleged accounting fraud at the bank.", 1, "G", "investigated"),
    ("The company denies the allegations; the investigation remains ongoing.", 1, "G", "denied / unresolved"),
    ("The board approved a new renewable-energy investment plan.", 0, "None", "not applicable"),
    ("Quarterly revenue exceeded analyst expectations.", 0, "None", "not applicable"),
    ("The company announced a target to cut operational emissions by 40%.", 0, "None", "not applicable"),
    ("A data breach exposed customer records; the cause has not yet been established.", 1, "S", "under investigation"),
], columns=["headline", "is_controversy", "pillar", "status"])
examples

,headline,is_controversy,pillar,status
0,Acme Energy was fined after a wastewater leak ...,1,E,confirmed
1,Workers at the supplier allege unsafe conditio...,1,S,alleged
2,The regulator opened an inquiry into alleged a...,1,G,investigated
3,The company denies the allegations; the invest...,1,G,denied / unresolved
4,The board approved a new renewable-energy inve...,0,None,not applicable
5,Quarterly revenue exceeded analyst expectations.,0,None,not applicable
6,The company announced a target to cut operatio...,0,None,not applicable
7,A data breach exposed customer records; the ca...,1,S,under investigation


## 4. Tokenization: text becomes model inputs

BERT tokenizers generally use **WordPiece subwords**: a rare or unseen word can be represented by multiple vocabulary pieces. The tokenizer adds special tokens, maps pieces to integer IDs, pads a batch to a common length, and creates an `attention_mask` where `1` marks real tokens and `0` marks padding. The mask prevents attention from treating padding as content.

Preserve punctuation, case behavior, and negation according to the model's tokenizer. Do not apply stemming or stop-word removal by default: it can alter syntax and remove meaningful terms such as `not`. Always inspect truncation for the chosen maximum length. Token IDs are vocabulary indices, not word meanings.

In [3]:
from transformers import AutoTokenizer

BASE_MODEL = "distilbert-base-uncased"
try:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    sample = "The supplier denies the allegations of unsafe working conditions."
    encoded = tokenizer(sample, truncation=True, max_length=64, return_tensors="pt")
    print("tokens:", tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]))
    print("input_ids:", encoded["input_ids"][0].tolist())
    print("attention_mask:", encoded["attention_mask"][0].tolist())
except Exception as exc:
    tokenizer = None
    print("Tokenizer/model files could not be downloaded or loaded:", type(exc).__name__, str(exc)[:300])

tokens: ['[CLS]', 'the', 'supplier', 'denies', 'the', 'allegations', 'of', 'unsafe', 'working', 'conditions', '.', '[SEP]']
input_ids: [101, 1996, 17024, 23439, 1996, 9989, 1997, 25135, 2551, 3785, 1012, 102]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## 5. Pretrained financial sentiment is a baseline, not the ESG target

FinBERT is a family of BERT models adapted to financial text. The checkpoint below is a **three-way financial sentiment** classifier; it returns probabilities over its configured labels. It does not directly predict `is_controversy`, pillar, allegation status, severity, or materiality. For example, positive language about a company can coexist with a serious environmental event, and neutral reporting can describe a major investigation.

We use it to inspect model behavior, not to create gold ESG labels. Check the model card, training data, intended use, license, and label mapping before adopting any checkpoint. Downloading requires access to the Hugging Face Hub unless model files are already cached.

In [4]:
from transformers import AutoModelForSequenceClassification

FINBERT_MODEL = "ProsusAI/finbert"
finbert_tokenizer = None
finbert_model = None
try:
    finbert_tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL)
    finbert_model = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL).to(DEVICE).eval()
    print("FinBERT label map:", finbert_model.config.id2label)
except Exception as exc:
    print("FinBERT unavailable (check network/cache and model compatibility):", type(exc).__name__, str(exc)[:300])

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

W1015 21:59:54.805000 50369 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FinBERT label map: {0: 'positive', 1: 'negative', 2: 'neutral'}


In [5]:
def classify_sentiment(texts, batch_size=8, max_length=128):
    """Return FinBERT sentiment scores; these are not ESG controversy probabilities."""
    if finbert_model is None or finbert_tokenizer is None:
        raise RuntimeError("Load the FinBERT checkpoint successfully before inference.")
    rows = []
    for start in range(0, len(texts), batch_size):
        batch = finbert_tokenizer(
            list(texts[start:start + batch_size]), padding=True, truncation=True,
            max_length=max_length, return_tensors="pt"
        ).to(DEVICE)
        with torch.inference_mode():
            probabilities = finbert_model(**batch).logits.softmax(dim=-1).cpu().numpy()
        for scores in probabilities:
            rows.append({finbert_model.config.id2label[i]: float(score) for i, score in enumerate(scores)})
    return pd.DataFrame(rows, index=list(range(len(texts))))

if finbert_model is not None:
    sentiment_scores = classify_sentiment(examples["headline"].tolist())
    display(pd.concat([examples[["headline", "is_controversy"]].reset_index(drop=True), sentiment_scores], axis=1))

,headline,is_controversy,positive,negative,neutral
0,Acme Energy was fined after a wastewater leak ...,1,0.019593,0.953048,0.027360
1,Workers at the supplier allege unsafe conditio...,1,0.009047,0.962643,0.028311
2,The regulator opened an inquiry into alleged a...,1,0.029430,0.749997,0.220572
3,The company denies the allegations; the invest...,1,0.036064,0.459555,0.504382
4,The board approved a new renewable-energy inve...,0,0.747170,0.008793,0.244038
5,Quarterly revenue exceeded analyst expectations.,0,0.953085,0.025290,0.021625
6,The company announced a target to cut operatio...,0,0.858637,0.026121,0.115243
7,A data breach exposed customer records; the ca...,1,0.010281,0.925778,0.063940


### Read the scores as evidence, not decisions

A softmax score is a model's relative score over its classes, not automatically a calibrated probability that the real-world event occurred. To estimate controversy risk, train and validate a classifier on human-labelled ESG examples. Keep separate fields for the predicted class, confidence, source, publication time, company/entity resolution, claim status, and analyst disposition. The headline examples above are illustrative; they are not a benchmark dataset.

## 6. Contextual embeddings and similarity

A frozen encoder can convert each text into a dense vector. Unlike TF-IDF, contextual token vectors depend on surrounding words. A common sentence representation is the **attention-mask-weighted mean** of the final hidden states: padding is excluded from the mean. This is a useful baseline, not a universal pooling strategy; `[CLS]` pooling, layer combinations, and task-trained sentence encoders may work better.

Cosine similarity can retrieve semantically related reports or near-duplicates for analyst triage. Similarity does not establish that two articles describe the same event, company, date, or claim. Verify entities and event details before deduplication, and do not use similarity as a truth score.

In [6]:
from transformers import AutoModel

embedding_tokenizer = None
embedding_model = None
try:
    embedding_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    embedding_model = AutoModel.from_pretrained(BASE_MODEL).to(DEVICE).eval()
    print("Loaded encoder:", BASE_MODEL)
except Exception as exc:
    print("Encoder unavailable (check network/cache):", type(exc).__name__, str(exc)[:300])


def mean_pool_embeddings(texts, batch_size=8, max_length=128):
    if embedding_model is None or embedding_tokenizer is None:
        raise RuntimeError("Load the base encoder successfully before embedding.")
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = embedding_tokenizer(
            list(texts[start:start + batch_size]), padding=True, truncation=True,
            max_length=max_length, return_tensors="pt"
        ).to(DEVICE)
        with torch.inference_mode():
            hidden = embedding_model(**batch).last_hidden_state
            mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        vectors.append(torch.nn.functional.normalize(pooled, p=2, dim=1).cpu().numpy())
    return np.vstack(vectors)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded encoder: distilbert-base-uncased


In [7]:
if embedding_model is not None:
    demo_texts = [
        "A wastewater leak contaminated a river near the company's facility.",
        "The company reported a chemical spill into a local waterway.",
        "The board declared a quarterly dividend for shareholders.",
    ]
    vectors = mean_pool_embeddings(demo_texts)
    similarity = vectors @ vectors.T  # vectors are unit-normalized
    display(pd.DataFrame(similarity, index=demo_texts, columns=demo_texts).round(3))

,A wastewater leak contaminated a river near the company's facility.,The company reported a chemical spill into a local waterway.,The board declared a quarterly dividend for shareholders.
A wastewater leak contaminated a river near the company's facility.,1.000,0.916,0.698
The company reported a chemical spill into a local waterway.,0.916,1.000,0.687
The board declared a quarterly dividend for shareholders.,0.698,0.687,1.000


## 7. Fine-tuning for ESG controversy classification

Fine-tuning requires labelled examples that match the intended decision. Define the annotation unit and policy first: whether credible allegations count as positive, how denials and unresolved investigations are handled, how multiple pillars are represented, and how supplier versus issuer events are attributed. Use multiple annotators on a sample, adjudicate disagreements, record provenance, and version the label guide.

The following recipe expects a CSV with columns `headline` and `is_controversy` (0/1). It is deliberately opt-in and does not use the tiny illustrative examples above as training data. For credible evaluation, prepare a substantially larger, representative dataset and split by event/entity and time where appropriate. Randomly splitting syndicated copies across train and test can produce inflated results.

In [8]:
from pathlib import Path
from sklearn.model_selection import train_test_split

DATA_PATH = Path("esg_headlines.csv")  # Replace with your annotated CSV path
RUN_FINE_TUNING = False  # Set True only after reviewing the data and split design

train_df = val_df = test_df = None
if DATA_PATH.exists():
    labeled = pd.read_csv(DATA_PATH)
    required = {"headline", "is_controversy"}
    if not required.issubset(labeled.columns):
        raise ValueError(f"CSV must contain columns: {sorted(required)}")
    labeled = labeled.dropna(subset=["headline", "is_controversy"]).copy()
    labeled["headline"] = labeled["headline"].astype(str)
    labeled["is_controversy"] = labeled["is_controversy"].astype(int)
    if not set(labeled["is_controversy"].unique()).issubset({0, 1}):
        raise ValueError("is_controversy must contain only 0 and 1")
    train_df, remainder = train_test_split(
        labeled, test_size=0.30, stratify=labeled["is_controversy"], random_state=SEED
    )
    val_df, test_df = train_test_split(
        remainder, test_size=0.50, stratify=remainder["is_controversy"], random_state=SEED
    )
    print("Split sizes:", len(train_df), len(val_df), len(test_df))
    print("Train positive rate:", round(train_df["is_controversy"].mean(), 3))
else:
    print(f"No training data found at {DATA_PATH}. Fine-tuning remains disabled.")

No training data found at esg_headlines.csv. Fine-tuning remains disabled.


In [9]:
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score

if RUN_FINE_TUNING and train_df is not None:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    classifier = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=2, id2label={0: "not_controversy", 1: "controversy"},
        label2id={"not_controversy": 0, "controversy": 1}
    ).to(DEVICE)

    def make_loader(frame, batch_size, shuffle):
        encoded = tokenizer(
            frame["headline"].tolist(), padding=True, truncation=True,
            max_length=128, return_tensors="pt"
        )
        dataset = TensorDataset(
            encoded["input_ids"], encoded["attention_mask"],
            torch.tensor(frame["is_controversy"].to_numpy(), dtype=torch.long)
        )
        return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

    train_loader = make_loader(train_df, batch_size=16, shuffle=True)
    val_loader = make_loader(val_df, batch_size=32, shuffle=False)
    counts = np.bincount(train_df["is_controversy"], minlength=2)
    if np.any(counts == 0):
        raise ValueError("Training split must contain examples from both classes")
    class_weights = torch.tensor(counts.sum() / (2 * counts), dtype=torch.float, device=DEVICE)
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(classifier.parameters(), lr=2e-5, weight_decay=0.01)

    def evaluate(loader):
        classifier.eval()
        actual, predicted = [], []
        with torch.inference_mode():
            for input_ids, attention_mask, labels in loader:
                output = classifier(
                    input_ids=input_ids.to(DEVICE), attention_mask=attention_mask.to(DEVICE)
                )
                actual.extend(labels.numpy().tolist())
                predicted.extend(output.logits.argmax(dim=-1).cpu().tolist())
        return f1_score(actual, predicted, average="macro", zero_division=0)

    for epoch in range(2):
        classifier.train()
        total_loss = 0.0
        for input_ids, attention_mask, labels in train_loader:
            input_ids, attention_mask, labels = input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = classifier(input_ids=input_ids, attention_mask=attention_mask).logits
            loss = loss_fn(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(classifier.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
        print(f"epoch {epoch + 1}: train loss={total_loss / len(train_loader):.4f}, val macro-F1={evaluate(val_loader):.3f}")
else:
    print("Fine-tuning not run. Set RUN_FINE_TUNING=True and provide a reviewed labelled CSV.")

Fine-tuning not run. Set RUN_FINE_TUNING=True and provide a reviewed labelled CSV.


### Train, validation, and test discipline

The example split is stratified and random for illustration only. In real news data, prefer an out-of-time test and group related stories by event, source, or issuer so close copies cannot appear in both training and test sets. Fit all learned preprocessing on training data only. Use validation data for model selection, early stopping, calibration, and decision-threshold selection; evaluate the final chosen system on the test set once.

For an imbalanced screening task, report the positive-class precision, recall, F1, confusion matrix, PR-AUC, and the base rate. Macro-F1 gives equal weight to the two classes; it does not encode the relative cost of false negatives and false positives. Choose thresholds from operational costs and analyst capacity, then validate them on realistic data.

In [ ]:
from sklearn.metrics import classification_report

In [10]:
if RUN_FINE_TUNING and train_df is not None:
    test_loader = make_loader(test_df, batch_size=32, shuffle=False)

    def positive_scores(loader):
        classifier.eval()
        scores = []
        with torch.inference_mode():
            for input_ids, attention_mask, _ in loader:
                logits = classifier(
                    input_ids=input_ids.to(DEVICE), attention_mask=attention_mask.to(DEVICE)
                ).logits
                scores.extend(logits.softmax(dim=-1)[:, 1].cpu().tolist())
        return np.asarray(scores)

    val_scores = positive_scores(val_loader)
    thresholds = np.linspace(0.05, 0.95, 91)
    val_f1 = [f1_score(val_df["is_controversy"], val_scores >= t, zero_division=0) for t in thresholds]
    best_threshold = float(thresholds[int(np.argmax(val_f1))])
    print(f"Validation-selected threshold={best_threshold:.2f}; val positive F1={max(val_f1):.3f}")

    test_scores = positive_scores(test_loader)
    test_pred = (test_scores >= best_threshold).astype(int)
    print("Held-out test report (run once after model selection):")
    print(classification_report(test_df["is_controversy"], test_pred, target_names=["not controversy", "controversy"], zero_division=0))
else:
    print("Test evaluation skipped because fine-tuning is disabled or no dataset was loaded.")

Test evaluation skipped because fine-tuning is disabled or no dataset was loaded.


## 8. Extensions: multiple labels and structured events

A headline can describe an environmental impact and a governance failure at the same time. In that case, represent pillars as a multi-hot vector (for example, `[1, 0, 1]` for E and G) and train independent sigmoid outputs with binary cross-entropy; do not force mutually exclusive softmax classes. Evaluate precision/recall and PR-AUC per label, plus macro averages, and define how rare labels are handled.

For event extraction, separate classification from span extraction and entity resolution. A practical pipeline may identify an event type, evidence span, issuer/supplier, location, event date, claim status, and source document. Validate extracted values against the original text and preserve provenance. Severity and materiality require explicit human-defined criteria; they cannot be inferred reliably from generic negative sentiment alone.

## 9. ESG error analysis and responsible use

Inspect false positives and false negatives by pillar, event type, allegation status, geography, company size, source, and time period. Typical failure modes include:
- **Negation and attribution:** "company denies" versus "regulator confirms"; quoted claims may not be the publisher's conclusion.
- **Entity confusion:** an event at a supplier or peer is attributed to the issuer.
- **Context and time:** historical controversies, resolved cases, and new allegations are mixed together.
- **Template and source leakage:** repeated wire stories or publisher phrasing appear across splits.
- **Label ambiguity:** policy disagreements about allegations, severity, and positive ESG initiatives are mistaken for model errors.
- **Distribution shift:** new regulations, terminology, regions, and event types change the input distribution.

Keep an evidence trail from prediction to source text and model version. Use confidence bands or abstention for uncertain cases, monitor drift and subgroup performance, and route consequential judgments to trained analysts. Do not treat model output as proof of misconduct or as an automated investment decision.

## 10. Study questions and exercises

1. Why can a financial sentiment model miss an ESG controversy written in neutral language?
2. How should the label policy treat a credible allegation that is denied but not yet resolved?
3. Why is `not` important in "the regulator found no wrongdoing"?
4. What leakage can occur if syndicated articles about one spill are randomly split?
5. When should pillar prediction use multi-label sigmoid outputs rather than softmax?
6. If false negatives are more costly, which metrics and validation choices matter beyond accuracy?
7. What is lost when a long annual report is truncated to the first 128 tokens? Propose a chunking and aggregation strategy.
8. Compare TF-IDF, a frozen BERT embedding, and a fine-tuned classifier for latency, data needs, interpretability, and expected failure modes.

**Exercises:** add examples for all three pillars and difficult negatives; build a TF-IDF logistic-regression baseline; compare it with a fine-tuned encoder using the same event/time-aware splits; inspect misclassifications; calibrate scores; and document the annotation policy and model limitations. Report results with confidence intervals or repeated temporal evaluations, not a single headline score.

## 11. Recommended end-to-end ESG modeling approach

1. **Specify the decision.** Define the text unit, target entity, positive class, allegation policy, and intended use. Keep controversy detection separate from sentiment, pillar, severity, and materiality.
2. **Create an auditable label set.** Write an annotation rubric, use multiple annotators on a sample, adjudicate disagreements, and retain source, date, and entity provenance. Preserve uncertainty instead of treating every allegation as a confirmed fact.
3. **Build a simple baseline.** Start with TF-IDF plus logistic regression or a linear SVM. This shows whether a lexical model is already competitive and provides a fast comparison.
4. **Design realistic evaluation.** Group related events and sources so syndicated copies cannot cross splits; use a chronological test when the real task predicts future coverage. Keep the final test period untouched during model and threshold selection.
5. **Compare pretrained approaches.** Evaluate the lexical baseline, frozen Transformer embeddings, and a fine-tuned encoder on the same labels and splits. Treat FinBERT sentiment as a sentiment baseline only, not as the controversy target.
6. **Measure the errors that matter.** Report class-specific precision and recall, F1, PR-AUC, calibration, and confusion matrices. Review errors by E/S/G pillar, allegation status, geography, source, and entity type.
7. **Choose an operating point.** Select thresholds on validation data using the cost of missed events, false alerts, and analyst capacity. Add an abstention or escalation path for uncertain or high-impact cases.
8. **Deploy as analyst support.** Return predictions with evidence text, source, date, entity, model version, and uncertainty. Monitor drift and subgroup performance, and update labels and models under governance rather than treating predictions as ground truth.

The workflow matters more than selecting a particular BERT checkpoint: label policy and evaluation design determine whether a high score means anything for the ESG decision.

## 12. BERT and FinBERT internals: vocabulary, layers, and training

### Vocabulary: the tokenizer is part of the model

A model's **vocabulary** is the fixed mapping between token strings and integer IDs. BERT's common English checkpoint, `bert-base-uncased`, uses WordPiece subwords and has 30,522 vocabulary entries, including special tokens such as `[CLS]`, `[SEP]`, `[MASK]`, and `[PAD]`. The uncased tokenizer lowercases ordinary text; a cased checkpoint preserves case. A word such as `decarbonisation` may be a single token or several pieces depending on the vocabulary. Company names, regulatory terms, and emissions abbreviations may also split into subwords.

The token ID is only an address into the model's embedding matrix, not a numeric measure of meaning. During training, the model learns a vector for every vocabulary ID. **The tokenizer and the model weights must match exactly:** if a tokenizer maps `spill` to ID 100 but a model learned ID 100 for a different piece, the model receives the wrong embedding. Load `AutoTokenizer` from the same checkpoint as the model; do not substitute a tokenizer just because it is also called BERT.

**FinBERT is a model/checkpoint name, not a single universal vocabulary.** The notebook uses `ProsusAI/finbert`, a financial-language BERT sequence-classification checkpoint. Many FinBERT checkpoints use a BERT-base-uncased vocabulary, but variants can differ. The inspection cell below compares token-to-ID mappings rather than relying on the name. `AutoTokenizer.from_pretrained(checkpoint)` is the safe default. If adding new ESG-specific tokens, you must resize the model's token-embedding matrix and train the new embeddings; this is usually unnecessary because subwords already represent unseen terms.

### Encoder layers: what is inside BERT?

The input token IDs are looked up in a **word-embedding table**. BERT adds **position embeddings** (where a token occurs) and **token-type embeddings** (which sentence/segment it belongs to, when used). The sum is normalized and passed through a stack of Transformer encoder layers.

A standard BERT-base encoder has 12 layers, hidden size 768, 12 self-attention heads, and a feed-forward intermediate size of 3,072, with roughly 110 million parameters. Each layer first lets tokens exchange contextual information through multi-head self-attention; then a position-wise feed-forward network transforms each token vector. Residual connections and layer normalization help information and gradients flow through the stack. Each attention head has its own learned projections; with hidden size 768 and 12 heads, each head works with a 64-dimensional slice before the heads are combined.

DistilBERT-base is a smaller distilled encoder with 6 layers, hidden size 768, 12 heads, and about 66 million parameters. It is faster and smaller, but it is not simply BERT with half the layers: it was trained with a distillation objective and omits some BERT components such as token-type embeddings. The exact architecture should be read from the checkpoint config, not guessed from its display name.

The final encoder output has shape `(batch_size, sequence_length, hidden_size)`. For sequence classification, a task head often reads the contextual vector at `[CLS]` (position zero) and maps it to class logits. For token classification, a head predicts a label for each token instead. The head is small compared with the encoder, but it changes the task the model is trained to perform.

In [11]:
from transformers import AutoConfig, AutoTokenizer

CHECKPOINTS = {
    "BERT base uncased": "bert-base-uncased",
    "FinBERT sentiment": "ProsusAI/finbert",
    "DistilBERT": "distilbert-base-uncased",
}
checkpoint_details = {}
for name, checkpoint in CHECKPOINTS.items():
    config = AutoConfig.from_pretrained(checkpoint)
    tok = AutoTokenizer.from_pretrained(checkpoint)
    checkpoint_details[name] = {"config": config, "tokenizer": tok}
    hidden_size = getattr(config, "hidden_size", getattr(config, "dim", "n/a"))
    print(
        f"{name}: model_type={config.model_type}, vocab={len(tok)}, "
        f"layers={getattr(config, 'num_hidden_layers', 'n/a')}, "
        f"hidden={hidden_size}, heads={getattr(config, 'num_attention_heads', 'n/a')}, "
        f"max_positions={getattr(config, 'max_position_embeddings', 'n/a')}"
    )

bert_vocab = checkpoint_details["BERT base uncased"]["tokenizer"].get_vocab()
finbert_vocab = checkpoint_details["FinBERT sentiment"]["tokenizer"].get_vocab()
print("BERT-base and ProsusAI FinBERT token-to-ID mappings identical:", bert_vocab == finbert_vocab)

terms = ["decarbonisation", "Scope 3 emissions", "greenwashing", "Acme Energy"]
for name in ["BERT base uncased", "FinBERT sentiment"]:
    tok = checkpoint_details[name]["tokenizer"]
    print(name, {term: tok.tokenize(term) for term in terms})

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BERT base uncased: model_type=bert, vocab=30522, layers=12, hidden=768, heads=12, max_positions=512
FinBERT sentiment: model_type=bert, vocab=30522, layers=12, hidden=768, heads=12, max_positions=512
DistilBERT: model_type=distilbert, vocab=30522, layers=6, hidden=768, heads=12, max_positions=512
BERT-base and ProsusAI FinBERT token-to-ID mappings identical: True
BERT base uncased {'decarbonisation': ['dec', '##ar', '##bon', '##isation'], 'Scope 3 emissions': ['scope', '3', 'emissions'], 'greenwashing': ['green', '##wash', '##ing'], 'Acme Energy': ['ac', '##me', 'energy']}
FinBERT sentiment {'decarbonisation': ['dec', '##ar', '##bon', '##isation'], 'Scope 3 emissions': ['scope', '3', 'emissions'], 'greenwashing': ['green', '##wash', '##ing'], 'Acme Energy': ['ac', '##me', 'energy']}


### Three training stages: do not confuse pretraining with classification

1. **Original pretraining:** BERT learns general language representations from a very large unlabeled corpus using masked language modeling (MLM). Roughly, selected tokens are hidden and the model learns to recover them from context. This is expensive to reproduce from scratch and needs a very large, diverse corpus.
2. **Domain-adaptive continued pretraining:** start from an existing pretrained encoder and continue its MLM training on raw, unlabeled financial or ESG text. This helps it adapt to domain language such as emissions scopes, stewardship, supply-chain terms, and regulatory phrasing. It still does not learn the label `controversy` because there are no controversy labels in this stage.
3. **Supervised task fine-tuning:** attach a task-specific classification layer and train on human-labelled examples. The encoder and the custom layer can both be updated. This is the stage that teaches the model the chosen ESG label policy.

**Important:** during MLM training the model needs an MLM prediction head, not the ESG classification head. The custom ESG layer is added after domain adaptation. The next cell is a real but opt-in continued-MLM training recipe. It expects one ESG document per line in `esg_unlabeled.txt`; it does not download or train anything unless `RUN_MLM_PRETRAIN=True`. One short corpus or one epoch is only a smoke test, not a reliable domain model. Hold out text for validation, monitor MLM loss/perplexity, deduplicate documents, respect data rights, and watch for catastrophic forgetting of general language.

In [15]:
from pathlib import Path
from torch.utils.data import DataLoader as MLMDataLoader, Dataset
from transformers import AutoModelForMaskedLM, DataCollatorForLanguageModeling

MLM_TEXT_PATH = Path("esg_unlabeled.txt")  # One raw document or paragraph per line
MLM_OUTPUT_DIR = Path("models/esg-bert-mlm")
MLM_START_CHECKPOINT = "bert-base-uncased"  # Model and tokenizer must match
RUN_MLM_PRETRAIN = False  # Opt in only with a reviewed corpus and adequate compute


class ESGTextDataset(Dataset):
    def __init__(self, texts, text_tokenizer, max_length=128):
        self.texts = texts
        self.text_tokenizer = text_tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        return self.text_tokenizer(
            self.texts[index], truncation=True, max_length=self.max_length,
            return_special_tokens_mask=True
        )


if RUN_MLM_PRETRAIN:
    if not MLM_TEXT_PATH.exists():
        raise FileNotFoundError(f"Place the unlabeled ESG corpus at {MLM_TEXT_PATH}")
    corpus_lines = [line.strip() for line in MLM_TEXT_PATH.read_text().splitlines() if line.strip()]
    if len(corpus_lines) < 2:
        raise ValueError("Provide multiple non-empty documents for MLM training")

    mlm_tokenizer = AutoTokenizer.from_pretrained(MLM_START_CHECKPOINT)
    mlm_model = AutoModelForMaskedLM.from_pretrained(MLM_START_CHECKPOINT).to(DEVICE)
    mlm_data = ESGTextDataset(corpus_lines, mlm_tokenizer)
    mlm_collator = DataCollatorForLanguageModeling(
        tokenizer=mlm_tokenizer, mlm=True, mlm_probability=0.15
    )
    mlm_loader = MLMDataLoader(mlm_data, batch_size=8, shuffle=True, collate_fn=mlm_collator)
    mlm_optimizer = torch.optim.AdamW(mlm_model.parameters(), lr=5e-5, weight_decay=0.01)

    mlm_model.train()
    running_loss = 0.0
    for step, batch in enumerate(mlm_loader, start=1):
        batch = {key: value.to(DEVICE) for key, value in batch.items()}
        mlm_optimizer.zero_grad(set_to_none=True)
        loss = mlm_model(**batch).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mlm_model.parameters(), max_norm=1.0)
        mlm_optimizer.step()
        running_loss += loss.item()
        if step % 50 == 0:
            print(f"step {step}: mean MLM loss={running_loss / 50:.4f}")
            running_loss = 0.0

    MLM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    mlm_model.save_pretrained(MLM_OUTPUT_DIR)
    mlm_tokenizer.save_pretrained(MLM_OUTPUT_DIR)
    print("Saved continued-pretraining checkpoint to", MLM_OUTPUT_DIR)
else:
    print("MLM pretraining skipped. Provide esg_unlabeled.txt and set RUN_MLM_PRETRAIN=True to opt in.")

MLM pretraining skipped. Provide esg_unlabeled.txt and set RUN_MLM_PRETRAIN=True to opt in.


### Add and train a custom ESG classification layer

After MLM domain adaptation, use the encoder as the starting representation and add a task head. The example below uses a small multilayer perceptron: take the final `[CLS]` vector, apply dropout, project it through a GELU hidden layer, then produce two logits (`not_controversy`, `controversy`). Cross-entropy compares those logits with the human labels. The optimizer updates both the encoder and the custom head; this is supervised fine-tuning, not MLM pretraining.

The encoder is loaded with `AutoModel`, which returns hidden states without a checkpoint-specific classification head. If the MLM output directory exists, the code uses that continued-pretraining checkpoint; otherwise it starts from the financial FinBERT encoder. Keep the tokenizer paired with whichever encoder is selected. The training block is off by default and uses the CSV splits created earlier. The validation F1 is only useful when the CSV is representative and the split is designed to prevent event/source leakage.

In [14]:
from sklearn.metrics import f1_score
from transformers import AutoModel, AutoTokenizer


class ESGCustomClassifier(torch.nn.Module):
    def __init__(self, encoder_name, num_labels=2, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        hidden_size = getattr(self.encoder.config, "hidden_size", None) or getattr(self.encoder.config, "dim")
        self.dropout = torch.nn.Dropout(dropout)
        self.hidden_layer = torch.nn.Linear(hidden_size, hidden_size // 2)
        self.classifier = torch.nn.Linear(hidden_size // 2, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_vector = outputs.last_hidden_state[:, 0, :]
        features = torch.nn.functional.gelu(self.hidden_layer(self.dropout(cls_vector)))
        return self.classifier(self.dropout(features))


CUSTOM_ENCODER = str(MLM_OUTPUT_DIR) if (MLM_OUTPUT_DIR / "config.json").exists() else "ProsusAI/finbert"
CUSTOM_OUTPUT_DIR = Path("models/esg-custom-classifier")
RUN_CUSTOM_HEAD_TRAINING = False

if RUN_CUSTOM_HEAD_TRAINING:
    if train_df is None or val_df is None:
        raise ValueError("Provide esg_headlines.csv and create the train/validation splits first")
    custom_tokenizer = AutoTokenizer.from_pretrained(CUSTOM_ENCODER)
    custom_model = ESGCustomClassifier(CUSTOM_ENCODER).to(DEVICE)

    def custom_loader(frame, batch_size, shuffle):
        encoded = custom_tokenizer(
            frame["headline"].tolist(), padding=True, truncation=True,
            max_length=128, return_tensors="pt"
        )
        tensors = TensorDataset(
            encoded["input_ids"], encoded["attention_mask"],
            torch.tensor(frame["is_controversy"].to_numpy(), dtype=torch.long)
        )
        return DataLoader(tensors, batch_size=batch_size, shuffle=shuffle)

    custom_train_loader = custom_loader(train_df, batch_size=16, shuffle=True)
    custom_val_loader = custom_loader(val_df, batch_size=32, shuffle=False)
    class_counts = np.bincount(train_df["is_controversy"], minlength=2)
    if np.any(class_counts == 0):
        raise ValueError("Training split must contain both controversy classes")
    custom_weights = torch.tensor(
        class_counts.sum() / (2 * class_counts), dtype=torch.float, device=DEVICE
    )
    custom_loss = torch.nn.CrossEntropyLoss(weight=custom_weights)
    custom_optimizer = torch.optim.AdamW(custom_model.parameters(), lr=2e-5, weight_decay=0.01)

    def custom_validation_f1(loader):
        custom_model.eval()
        actual, predicted = [], []
        with torch.inference_mode():
            for input_ids, attention_mask, labels in loader:
                logits = custom_model(input_ids.to(DEVICE), attention_mask.to(DEVICE))
                actual.extend(labels.tolist())
                predicted.extend(logits.argmax(dim=-1).cpu().tolist())
        return f1_score(actual, predicted, average="macro", zero_division=0)

    for epoch in range(2):
        custom_model.train()
        epoch_loss = 0.0
        for input_ids, attention_mask, labels in custom_train_loader:
            input_ids, attention_mask, labels = input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)
            custom_optimizer.zero_grad(set_to_none=True)
            logits = custom_model(input_ids, attention_mask)
            loss = custom_loss(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(custom_model.parameters(), max_norm=1.0)
            custom_optimizer.step()
            epoch_loss += loss.item()
        print(
            f"epoch {epoch + 1}: train loss={epoch_loss / len(custom_train_loader):.4f}, "
            f"validation macro-F1={custom_validation_f1(custom_val_loader):.3f}"
        )

    CUSTOM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    custom_model.encoder.save_pretrained(CUSTOM_OUTPUT_DIR)
    custom_tokenizer.save_pretrained(CUSTOM_OUTPUT_DIR)
    torch.save(
        {"hidden_layer": custom_model.hidden_layer.state_dict(), "classifier": custom_model.classifier.state_dict()},
        CUSTOM_OUTPUT_DIR / "custom_head.pt"
    )
    print("Saved encoder, tokenizer, and custom head to", CUSTOM_OUTPUT_DIR)
else:
    print("Custom-head training skipped. Set RUN_CUSTOM_HEAD_TRAINING=True after reviewing a labelled CSV.")

Custom-head training skipped. Set RUN_CUSTOM_HEAD_TRAINING=True after reviewing a labelled CSV.
